# Module 1 — Exploratory Data Analysis

**Goal:** Understand the shape, quality, and patterns in our three datasets before building any models.

Good EDA tells you:
- How big the data is and whether it fits in memory
- Which columns have missing values and what to do about them
- What the distributions look like (skewed? bimodal? lots of outliers?)
- What business patterns emerge (peak shopping days, top categories, loyal vs casual customers)

**Datasets we're exploring:**
1. **Instacart** — real grocery orders from 206K customers (3.4M orders)
2. **M5** — hierarchical daily sales across 3,049 Walmart products
3. **Open Food Facts** — product catalogue for our RAG knowledge base

In [ ]:
import sys
sys.path.insert(0, '..')  # lets us import from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

from src.data.loader import load_orders, load_products, load_order_items, load_m5_sales
from src.config import ORDERS_PATH, ORDER_ITEMS_PATH, PRODUCTS_PATH

# Matplotlib styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Libraries loaded ✓')

## Part 1: Instacart — Order Patterns

### 1.1 What does the data look like?

The Instacart dataset has three tables that join together:

```
orders          order_items         products
----------      -----------         --------
order_id  ───── order_id            product_id ──── product_id
user_id         product_id ─────────product_name
order_dow       reordered           aisle
order_hour      add_to_cart_order   department
days_since_prior
```

This is a **relational data model** — data is split across multiple tables to avoid repetition, and joined when you need a complete picture.

In [ ]:
orders      = load_orders()
products    = load_products()
order_items = load_order_items()

print('=== ORDERS ===')
print(f'Shape: {orders.shape}')
print(orders.dtypes)
print()
orders.head(3)

In [ ]:
print('=== PRODUCTS ===')
print(f'Shape: {products.shape}')
products.head(3)

In [ ]:
print('=== ORDER ITEMS ===')
print(f'Shape: {order_items.shape}')
print(f'Reorder rate: {order_items["reordered"].mean():.1%}')  # % of items that are reorders
order_items.head(3)

### 1.2 When do people shop?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Orders by day of week (0=Saturday in Instacart's encoding)
dow_labels = ['Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri']
orders['order_dow'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue'
)
axes[0].set_xticklabels(dow_labels, rotation=0)
axes[0].set_title('Orders by Day of Week')
axes[0].set_ylabel('Number of Orders')

# Orders by hour of day
orders['order_hour_of_day'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='coral'
)
axes[1].set_title('Orders by Hour of Day')
axes[1].set_ylabel('Number of Orders')

plt.tight_layout()
plt.savefig('../data/processed/eda_order_timing.png', dpi=150, bbox_inches='tight')
plt.show()

# Key insight: weekends and mid-morning are peak shopping times
# This matters for demand forecasting — sales spike on weekends

### 1.3 How many items per basket?

**Basket size** is a key retail metric — it affects average order value and picking efficiency.

In [ ]:
# Count items per order using DuckDB SQL
# This is much faster than a pandas groupby on 33M rows
basket_sizes = duckdb.query(f"""
    SELECT order_id, COUNT(*) as basket_size
    FROM '{ORDER_ITEMS_PATH}'
    GROUP BY order_id
""").df()

print(basket_sizes['basket_size'].describe())

basket_sizes['basket_size'].clip(upper=40).plot(
    kind='hist', bins=40, figsize=(10, 4),
    title='Distribution of Basket Size (items per order)',
    xlabel='Items per order'
)
plt.savefig('../data/processed/eda_basket_size.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.4 Top products and departments

In [ ]:
# Join order items with product names via DuckDB SQL
top_products = duckdb.query(f"""
    SELECT
        p.product_name,
        p.department,
        COUNT(*) as order_count,
        AVG(oi.reordered) as reorder_rate
    FROM '{ORDER_ITEMS_PATH}' oi
    JOIN '{PRODUCTS_PATH}' p USING (product_id)
    GROUP BY p.product_name, p.department
    ORDER BY order_count DESC
    LIMIT 15
""").df()

print(top_products.to_string(index=False))

In [ ]:
# Sales by department
dept_sales = duckdb.query(f"""
    SELECT p.department, COUNT(*) as n_items
    FROM '{ORDER_ITEMS_PATH}' oi
    JOIN '{PRODUCTS_PATH}' p USING (product_id)
    GROUP BY department
    ORDER BY n_items DESC
""").df()

dept_sales.plot(
    x='department', y='n_items', kind='barh',
    figsize=(10, 6), legend=False,
    title='Items Purchased by Department'
)
plt.xlabel('Total Items')
plt.tight_layout()
plt.savefig('../data/processed/eda_departments.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.5 Reorder behaviour

The `reordered` flag is our target variable for the **reorder classifier** model (Module 2). Understanding its distribution tells us if we have a class imbalance problem.

**Class imbalance:** If 90% of observations are class 0 (not reordered), a model that always predicts 0 gets 90% accuracy without learning anything. We need to check this.

In [ ]:
reorder_rate = order_items['reordered'].mean()
print(f'Overall reorder rate: {reorder_rate:.1%}')
print(f'This means {reorder_rate:.1%} of items in a basket have been bought before')
print(f'Class imbalance: {reorder_rate:.2f} vs {1-reorder_rate:.2f} — manageable')

## Part 2: M5 — Sales Time Series

The M5 dataset is structured differently from Instacart. Instead of individual customer orders, it shows **daily aggregate sales** at the store level. This is what Woolworths would actually use for inventory planning.

**Why time series is different from regular ML:**
In standard ML, rows are independent. In time series, row at time *t* is correlated with row at time *t-1*. If you randomly shuffle and split train/test, you'll leak future data into training — so we always split by time (train on older data, test on newer).

In [ ]:
m5 = load_m5_sales()
print(f'M5 shape: {m5.shape}')
print(f'Date range: {m5["date"].min()} → {m5["date"].max()}')
print(f'Products: {m5["item_id"].nunique()}')
print(f'Stores: {m5["store_id"].nunique()}')
m5.head()

In [ ]:
# Aggregate daily sales across all products and stores
daily_sales = m5.groupby('date')['sales'].sum().reset_index()

plt.figure(figsize=(14, 4))
plt.plot(daily_sales['date'], daily_sales['sales'], linewidth=0.8, alpha=0.8)
plt.title('M5: Total Daily Sales Across All Products and Stores')
plt.xlabel('Date')
plt.ylabel('Total Units Sold')
plt.tight_layout()
plt.savefig('../data/processed/eda_m5_daily_sales.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Seasonal patterns: average sales by month
m5['month'] = pd.to_datetime(m5['date']).dt.month
monthly = m5.groupby('month')['sales'].mean()

month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly.plot(kind='bar', figsize=(10, 4),
             title='Average Daily Sales by Month (M5)',
             xlabel='Month')
plt.xticks(range(12), month_labels, rotation=0)
plt.ylabel('Avg Units Sold per Day')
plt.tight_layout()
plt.show()

## Part 3: Open Food Facts — Product Catalogue

This dataset will be our **RAG knowledge base**. We need to understand its coverage and data quality before embedding it into a vector database.

**Key questions for a RAG knowledge base:**
- How complete are the text fields we'll embed? (Empty descriptions = useless embeddings)
- Are there enough products to make retrieval meaningful?
- What categories exist? (We'll use these as metadata filters)

In [ ]:
from src.data.loader import load_off_products

off = load_off_products()
print(f'Products: {len(off):,}')
print(f'\nMissing values:')
print(off.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
# Top categories
top_cats = (
    off['main_category_en']
    .dropna()
    .value_counts()
    .head(20)
)
top_cats.plot(kind='barh', figsize=(10, 6),
              title='Top 20 Product Categories (Open Food Facts — Australia)')
plt.tight_layout()
plt.savefig('../data/processed/eda_off_categories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample product records to understand text quality
sample = off[['product_name', 'brands', 'main_category_en', 'nutriscore_grade']].dropna().sample(5)
print(sample.to_string(index=False))

## Summary

Key findings from EDA:

| Dataset | Rows | Key insight |
|---|---|---|
| Instacart orders | ~3.4M | Weekend/mid-morning peaks; avg basket ~10 items |
| Order items | ~33M | ~60% reorder rate — moderate class balance |
| M5 sales | ~50M | Clear seasonal pattern; sparse (many zero-sales days) |
| Open Food Facts | ~50K (AU) | Good coverage; ingredients text present for most |

**Next:** Module 2 — build the four ML models (demand forecasting, reorder classifier, customer segmentation, price elasticity)